In [1]:
# compile the target dysregulated genes/proteins for the 7 causal network regulators of interest for each timepoint from proteomics data and for each significant z-score
## regulators: {'AREG', 'ATM', 'CUL4B', 'MYC', 'Pkg', 'ROCK', 'Rac'}
import pandas as pd
import numpy as np

In [4]:
ctable = pd.read_excel('../results/IPA_downstream_analysis/PROT_causalreg.xlsx',
                       sheet_name=None)

In [10]:
regs = set(ctable['shared']['Master Regulator'])
regs

{'AREG', 'ATM', 'CUL4B', 'MYC', 'Pkg', 'ROCK', 'Rac'}

In [63]:
# read in IPA database annotations/information for gene symbol, uniprot, etc. annotations
anno = pd.read_csv('../data/IPA_export/ipa_anno.txt',sep='\t',header=1)
anno.head()

,Expr Log Ratio,Expr p-value,ID,Flags,Symbol,Entrez Gene Name,Location,Type(s),Drug(s),Unnamed: 9
0,-0.111,0.000007,P01023,,A2M,alpha-2-macroglobulin,Extracellular Space,transporter,,NaN
1,0.253,0.000143,Q9NRG9,,AAAS,aladin WD repeat nucleoporin,Nucleus,other,,NaN
2,-0.820,0.000519,Q86V21,,AACS,acetoacetyl-CoA synthetase,Cytoplasm,enzyme,,NaN
3,-0.423,0.195000,A0A096LP25,D,AAK1,AP2 associated kinase 1,Cytoplasm,kinase,"LP-935509, SM1-71",NaN
4,-0.316,0.000545,Q2M2I8,D,AAK1,AP2 associated kinase 1,Cytoplasm,kinase,"LP-935509, SM1-71",NaN


In [106]:
cdict = {}
for r in regs:
    cdict[r] = {}


for t in [i for i in [*ctable] if 'sig' in i]:
    time = t[:t.index('_PROT')]
    
    for r in regs:
        regdf = ctable[t][ctable[t]['Master Regulator'] == r]
        
        for ind, row in regdf.iterrows():
            zscore = np.round(row['Activation z-score'],3)
            genes = row['Target Molecules in Dataset'].split(',')
            alsoreg = row['Participating regulators']
            anno_map = [','.join([i for i in anno[anno['Symbol'] == g]['ID'].values]) for g in genes]
            
            cdict[r][time+'_z'+str(zscore)+'_symbol'] = genes
            cdict[r][time+'_z'+str(zscore)+'_reg'] = [1 if i in alsoreg else 0 for i in genes]
            cdict[r][time+'_z'+str(zscore)+'_uniprot'] = anno_map

In [107]:
with pd.ExcelWriter('../results/IPA_downstream_analysis/PROT_sharedreg.xlsx') as writer:
    for r in regs:
        rdf = pd.DataFrame.from_dict(cdict[r],orient='index').T
        rdf.to_excel(writer, sheet_name=r)
writer.save()